# Fashion Retail Sales 

A boutique-style walkthrough of **3,400 transactions** from a fashion retailer: data quality, feature engineering, customer value, and time trends.

**Questions this notebook answers**
1. Where is revenue concentrated — items, categories, payment mix?
2. Which customers look loyal vs. at risk (RFM + segments)?
3. How do sales and ratings move through the year?
4. What should merchandising and CRM do next?

**Source:** `Fashion_Retail_Sales.csv` · 

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

DATA_PATH = Path("Fashion_Retail_Sales.csv")

# Boutique palette
ROSE = "#C97B84"
BLUSH = "#E8B4C8"
PEACH = "#F3C6B0"
GOLD = "#D4A574"
SAGE = "#A8C5B0"
LILAC = "#C9B8D9"
CREAM = "#FFF8F4"
INK = "#3D2C2E"
MUTED = "#8A6E73"
PALETTE = [ROSE, SAGE, GOLD, LILAC, PEACH, BLUSH, "#7BA3C9", "#E6A57E"]

sns.set_theme(
    style="whitegrid",
    rc={
        "axes.facecolor": CREAM,
        "figure.facecolor": CREAM,
        "axes.edgecolor": "#E8D5D0",
        "axes.labelcolor": INK,
        "text.color": INK,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "grid.color": "#F0E4DF",
        "font.family": "DejaVu Sans",
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
    },
)
sns.set_palette(PALETTE)


def money(ax, axis="y"):
    fmt = mtick.FuncFormatter(lambda x, _: f"${x:,.0f}")
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(fmt)


def cute_bar(ax):
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    ax.spines["left"].set_color("#E8D5D0")
    ax.spines["bottom"].set_color("#E8D5D0")


print("Ready ·", DATA_PATH.resolve())

## 1. Load & first look

In [ ]:
raw = pd.read_csv(DATA_PATH)
print(f"{len(raw):,} rows × {raw.shape[1]} columns")
raw.head()

In [ ]:
raw.info()

In [ ]:
raw.describe(include="all")

## 2. Data quality

Expect missing **purchase amounts** and **review ratings**. Amounts also jump from a hard cap around **$200** to four-digit values — treat those as scale errors (likely cents, or an extra zero).

In [ ]:
quality = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "missing_%": (raw.isna().mean() * 100).round(2),
    "unique": raw.nunique(),
})
quality

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

miss = (raw.isna().mean() * 100).sort_values(ascending=True)
colors = [ROSE if v > 0 else SAGE for v in miss]
axes[0].barh(miss.index, miss.values, color=colors, height=0.62)
axes[0].set_title("Missing values by column")
axes[0].set_xlabel("% of rows")
cute_bar(axes[0])

amt = raw["Purchase Amount (USD)"].dropna()
axes[1].hist(amt[amt <= 250], bins=30, color=BLUSH, edgecolor="white")
axes[1].axvline(amt.median(), color=ROSE, ls="--", lw=2, label=f"median ${amt.median():.0f}")
n_hi = (amt > 200).sum()
axes[1].set_title(f"Purchase amount (typical range) · {n_hi} extreme values hidden")
axes[1].set_xlabel("USD")
axes[1].legend(frameon=False)
cute_bar(axes[1])

plt.tight_layout()
plt.show()

print("Duplicate rows:", raw.duplicated().sum())
print("Amount > $200 (scale suspects):", int((raw["Purchase Amount (USD)"] > 200).sum()))

## 3. Clean & engineer

In [ ]:
CATEGORY_MAP = {
    "Tops": [
        "Blouse", "Camisole", "Flannel Shirt", "Hoodie", "Polo Shirt", "Sweater",
        "T-shirt", "Tank Top", "Tunic", "Vest", "Cardigan", "Kimono",
    ],
    "Bottoms": ["Jeans", "Leggings", "Pants", "Shorts", "Skirt", "Trousers"],
    "Outerwear": ["Blazer", "Coat", "Jacket", "Poncho", "Raincoat", "Trench Coat"],
    "Dresses & sets": ["Dress", "Jumpsuit", "Onesie", "Overalls", "Pajamas", "Romper", "Swimsuit"],
    "Footwear": ["Boots", "Flip-Flops", "Loafers", "Sandals", "Slippers", "Sneakers"],
    "Accessories": [
        "Backpack", "Belt", "Bowtie", "Gloves", "Handbag", "Hat", "Scarf",
        "Socks", "Sun Hat", "Sunglasses", "Tie", "Umbrella", "Wallet",
    ],
}
ITEM_TO_CATEGORY = {item: cat for cat, items in CATEGORY_MAP.items() for item in items}

df = raw.copy()
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"[()]", "", regex=True)
    .str.replace(" ", "_")
)
df = df.rename(columns={
    "customer_reference_id": "customer_id",
    "purchase_amount_usd": "purchase_amount_raw",
    "date_purchase": "purchase_date",
})

df["purchase_date"] = pd.to_datetime(df["purchase_date"], dayfirst=True, errors="coerce")
assert df["purchase_date"].isna().sum() == 0, "Unparseable dates"

df["amount_scale_flag"] = df["purchase_amount_raw"] > 200
df["purchase_amount"] = np.where(
    df["amount_scale_flag"],
    df["purchase_amount_raw"] / 100.0,
    df["purchase_amount_raw"],
)

item_amt_median = df.groupby("item_purchased")["purchase_amount"].median()
item_rat_median = df.groupby("item_purchased")["review_rating"].median()
df["purchase_amount"] = df["purchase_amount"].fillna(
    df["item_purchased"].map(item_amt_median)
).fillna(df["purchase_amount"].median())
df["review_rating"] = df["review_rating"].fillna(
    df["item_purchased"].map(item_rat_median)
).fillna(df["review_rating"].median())

df["category"] = df["item_purchased"].map(ITEM_TO_CATEGORY)
unmapped = df.loc[df["category"].isna(), "item_purchased"].unique()
if len(unmapped):
    print("Unmapped items (assigned Other):", unmapped)
    df["category"] = df["category"].fillna("Other")

df["year"] = df["purchase_date"].dt.year
df["month"] = df["purchase_date"].dt.month
df["month_name"] = df["purchase_date"].dt.strftime("%b")
df["year_month"] = df["purchase_date"].dt.to_period("M").astype(str)
df["weekday"] = df["purchase_date"].dt.day_name()
df["week"] = df["purchase_date"].dt.isocalendar().week.astype(int)

season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Fall", 10: "Fall", 11: "Fall",
}
df["season"] = df["month"].map(season_map)

df["spend_tier"] = pd.qcut(
    df["purchase_amount"], q=4, labels=["Value", "Core", "Premium", "Luxury"]
)
df["rating_band"] = pd.cut(
    df["review_rating"],
    bins=[0, 2, 3, 4, 5],
    labels=["Poor (≤2)", "Fair (2–3)", "Good (3–4)", "Excellent (4–5)"],
    include_lowest=True,
)

snapshot = df["purchase_date"].max() + pd.Timedelta(days=1)
rfm = (
    df.groupby("customer_id")
    .agg(
        recency_days=("purchase_date", lambda s: (snapshot - s.max()).days),
        frequency=("customer_id", "size"),
        monetary=("purchase_amount", "sum"),
        avg_ticket=("purchase_amount", "mean"),
        avg_rating=("review_rating", "mean"),
        last_purchase=("purchase_date", "max"),
    )
    .reset_index()
)
df = df.merge(
    rfm[["customer_id", "recency_days", "frequency", "monetary"]],
    on="customer_id",
    how="left",
)

print(df.isna().sum().to_string())
print("\nDate span:", df["purchase_date"].min().date(), "→", df["purchase_date"].max().date())
print("Scale flags corrected:", int(df["amount_scale_flag"].sum()))
df.head()

## 4. Executive snapshot

In [ ]:
kpi = {
    "Revenue": f"${df['purchase_amount'].sum():,.0f}",
    "Orders": f"{len(df):,}",
    "Customers": f"{df['customer_id'].nunique():,}",
    "AOV": f"${df['purchase_amount'].mean():.1f}",
    "Median ticket": f"${df['purchase_amount'].median():.1f}",
    "Avg rating": f"{df['review_rating'].mean():.2f} / 5",
    "Orders / customer": f"{df.groupby('customer_id').size().mean():.1f}",
}

fig, ax = plt.subplots(figsize=(12, 2.4))
ax.set_xlim(0, len(kpi))
ax.set_ylim(0, 1)
ax.axis("off")
ax.set_title("Boutique scoreboard", loc="left", pad=8)
for i, (label, value) in enumerate(kpi.items()):
    ax.text(i + 0.5, 0.62, value, ha="center", va="center", fontsize=15, fontweight="bold", color=ROSE)
    ax.text(i + 0.5, 0.28, label, ha="center", va="center", fontsize=9, color=MUTED)
plt.tight_layout()
plt.show()
pd.Series(kpi, name="value").to_frame()

## 5. Mix: category, items, payment

In [ ]:
cat = (
    df.groupby("category", as_index=False)
    .agg(orders=("customer_id", "size"), revenue=("purchase_amount", "sum"), avg_rating=("review_rating", "mean"))
    .sort_values("revenue", ascending=False)
)

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4))

wedges, *_ = axes[0].pie(
    cat["revenue"],
    labels=cat["category"],
    colors=PALETTE[: len(cat)],
    startangle=90,
    wedgeprops=dict(width=0.48, edgecolor=CREAM, linewidth=3),
    textprops=dict(color=INK, fontsize=8),
)
axes[0].set_title("Revenue by category")

pay = df.groupby("payment_method")["purchase_amount"].sum()
axes[1].pie(
    pay,
    labels=pay.index,
    autopct="%1.1f%%",
    colors=[ROSE, SAGE],
    startangle=90,
    wedgeprops=dict(width=0.48, edgecolor=CREAM, linewidth=3),
    textprops=dict(color=INK, fontsize=9),
)
axes[1].set_title("Revenue by payment method")

top_items = (
    df.groupby("item_purchased")["purchase_amount"].sum().nlargest(12).sort_values()
)
axes[2].barh(top_items.index, top_items.values, color=sns.color_palette("ch:s=-.2,r=.6", n_colors=12), height=0.7)
axes[2].set_title("Top 12 items by revenue")
money(axes[2])
cute_bar(axes[2])

plt.tight_layout()
plt.show()
cat.round({"revenue": 0, "avg_rating": 2})

In [ ]:
item_stats = (
    df.groupby(["category", "item_purchased"], as_index=False)
    .agg(
        orders=("customer_id", "size"),
        revenue=("purchase_amount", "sum"),
        avg_ticket=("purchase_amount", "mean"),
        avg_rating=("review_rating", "mean"),
    )
)

fig, ax = plt.subplots(figsize=(11, 6.2))
sns.scatterplot(
    data=item_stats,
    x="orders",
    y="avg_rating",
    hue="category",
    size="revenue",
    sizes=(40, 420),
    palette=PALETTE[: item_stats["category"].nunique()],
    ax=ax,
    alpha=0.9,
)
ax.axhline(df["review_rating"].mean(), color=MUTED, ls=":", lw=1)
ax.axvline(item_stats["orders"].median(), color=MUTED, ls=":", lw=1)
ax.set_title("Item map · volume vs love (bubble = revenue)")
ax.set_xlabel("Orders")
ax.set_ylabel("Average review")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
cute_bar(ax)
plt.tight_layout()
plt.show()

stars = item_stats.sort_values("avg_rating", ascending=False).head(8)
risks = item_stats.sort_values("avg_rating").head(8)
print("Highest-rated items")
display(stars.round(2))
print("Lowest-rated items (quality / fit risk)")
display(risks.round(2))

## 6. Price & ratings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))

order = ["Accessories", "Tops", "Bottoms", "Outerwear", "Dresses & sets", "Footwear"]
order = [c for c in order if c in df["category"].unique()]
sns.violinplot(
    data=df, x="category", y="purchase_amount", hue="category", order=order,
    inner="quartile", palette=PALETTE, cut=0, ax=axes[0], legend=False,
)
axes[0].set_title("Ticket size by category")
axes[0].tick_params(axis="x", rotation=20)
axes[0].set_xlabel("")
axes[0].set_ylabel("Purchase amount (USD)")
cute_bar(axes[0])

sns.kdeplot(
    data=df, x="review_rating", hue="payment_method",
    fill=True, common_norm=False, alpha=0.45, ax=axes[1],
    palette=[ROSE, SAGE],
)
axes[1].set_title("Rating density by payment method")
axes[1].set_xlabel("Review rating")
cute_bar(axes[1])

plt.tight_layout()
plt.show()

tier = df.groupby("spend_tier").agg(
    orders=("customer_id", "size"),
    revenue=("purchase_amount", "sum"),
    avg_rating=("review_rating", "mean"),
)
tier.assign(revenue_share=lambda x: (x["revenue"] / x["revenue"].sum() * 100).round(1))

## 7. Time: seasonality & calendar

In [ ]:
monthly = (
    df.groupby("year_month", as_index=False)
    .agg(revenue=("purchase_amount", "sum"), orders=("customer_id", "size"), avg_rating=("review_rating", "mean"))
    .sort_values("year_month")
)

fig, ax1 = plt.subplots(figsize=(12, 4.4))
ax1.fill_between(monthly["year_month"], monthly["revenue"], color=BLUSH, alpha=0.55)
ax1.plot(monthly["year_month"], monthly["revenue"], color=ROSE, lw=2.4, marker="o", ms=5)
ax1.set_ylabel("Revenue (USD)")
money(ax1)
ax2 = ax1.twinx()
ax2.plot(monthly["year_month"], monthly["avg_rating"], color=SAGE, lw=2, marker="s", ms=4, label="Avg rating")
ax2.set_ylabel("Avg rating")
ax2.set_ylim(2.4, 3.6)
ax1.set_title("Monthly revenue with overlay of average rating")
ax1.tick_params(axis="x", rotation=45)
cute_bar(ax1)
ax2.spines["top"].set_visible(False)
plt.tight_layout()
plt.show()

print("Note: Oct 2023 looks truncated")

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heat = (
    df.pivot_table(index="weekday", columns="month", values="purchase_amount", aggfunc="sum")
    .reindex(weekday_order)
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

sns.heatmap(heat, cmap="RdPu", ax=axes[0], linewidths=0.6, linecolor=CREAM)
axes[0].set_title("Revenue heatmap · weekday × month")
axes[0].set_xlabel("Month number")

season_cat = pd.crosstab(df["season"], df["category"], values=df["purchase_amount"], aggfunc="sum")
season_order = [s for s in ["Winter", "Spring", "Summer", "Fall"] if s in season_cat.index]
season_cat = season_cat.reindex(season_order)
season_cat.plot(kind="bar", stacked=True, ax=axes[1], color=PALETTE[: season_cat.shape[1]], width=0.72, edgecolor=CREAM)
axes[1].set_title("Seasonal revenue mix")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=0)
money(axes[1])
axes[1].legend(frameon=False, fontsize=8, loc="upper right")
cute_bar(axes[1])

plt.tight_layout()
plt.show()

## 8. Customers — RFM & segments

In [ ]:
feat = rfm[["recency_days", "frequency", "monetary"]].copy()
X = StandardScaler().fit_transform(feat)

inertias = []
for k in range(2, 8):
    inertias.append(KMeans(n_clusters=k, random_state=42, n_init=10).fit(X).inertia_)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=25)
rfm["cluster"] = kmeans.fit_predict(X)

# Name clusters from their profile (high money / high freq / low recency = VIPs)
profile = rfm.groupby("cluster")[["recency_days", "frequency", "monetary"]].mean()
score = (
    (profile["monetary"].rank(pct=True))
    + (profile["frequency"].rank(pct=True))
    + (1 - profile["recency_days"].rank(pct=True))
)
labels = {i: name for i, name in zip(score.sort_values(ascending=False).index, ["VIP muses", "Loyal regulars", "Steady core", "Cooling off"])}
rfm["segment"] = rfm["cluster"].map(labels)

seg = (
    rfm.groupby("segment")
    .agg(
        customers=("customer_id", "size"),
        revenue=("monetary", "sum"),
        avg_orders=("frequency", "mean"),
        avg_recency=("recency_days", "mean"),
        avg_rating=("avg_rating", "mean"),
    )
    .reindex(["VIP muses", "Loyal regulars", "Steady core", "Cooling off"])
)

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.3))
axes[0].plot(range(2, 8), inertias, color=ROSE, marker="o")
axes[0].set_title("K-means elbow")
axes[0].set_xlabel("k")
cute_bar(axes[0])

sns.scatterplot(
    data=rfm, x="frequency", y="monetary", hue="segment",
    palette=[ROSE, SAGE, GOLD, LILAC], s=55, ax=axes[1], alpha=0.85,
)
axes[1].set_title("Frequency vs customer revenue")
money(axes[1])
axes[1].legend(frameon=False, fontsize=8)
cute_bar(axes[1])

rev_share = rfm.groupby("segment")["monetary"].sum().reindex(seg.index)
axes[2].bar(rev_share.index, rev_share.values, color=[ROSE, SAGE, GOLD, LILAC], width=0.7)
axes[2].set_title("Revenue by segment")
axes[2].tick_params(axis="x", rotation=20)
money(axes[2])
cute_bar(axes[2])

plt.tight_layout()
plt.show()
seg.round(1)

In [ ]:
pref = (
    df.merge(rfm[["customer_id", "segment"]], on="customer_id")
    .groupby(["segment", "category"])["purchase_amount"].sum()
    .unstack(fill_value=0)
)
pref_share = pref.div(pref.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(10, 4.2))
sns.heatmap(pref_share, cmap="RdPu", annot=True, fmt=".0%", ax=ax, linewidths=1, linecolor=CREAM)
ax.set_title("What each segment buys (share of their spend)")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 9. Associations worth merchandising



In [ ]:
baskets = (
    df.groupby("customer_id")["item_purchased"]
    .apply(lambda s: set(s))
)
n_cust = len(baskets)
support = df.groupby("item_purchased")["customer_id"].nunique() / n_cust

pairs = []
items = support.nlargest(18).index.tolist()
for i, a in enumerate(items):
    for b in items[i + 1 :]:
        both = sum(1 for basket in baskets if a in basket and b in basket) / n_cust
        lift = both / (support[a] * support[b]) if support[a] * support[b] else np.nan
        pairs.append({"a": a, "b": b, "co_buyers": both, "lift": lift})

pair_df = pd.DataFrame(pairs).sort_values("lift", ascending=False)
print("Highest-lift pairs among popular items (cross-sell)")
pair_df.head(12).round(3)

## 10. Findings & recommendations



In [ ]:
top_cat = cat.iloc[0]
pay_mix = (df.groupby("payment_method")["purchase_amount"].sum() / df["purchase_amount"].sum() * 100).round(1)
best_month = monthly.loc[monthly["revenue"].idxmax()]
vip = seg.loc["VIP muses"] if "VIP muses" in seg.index else seg.iloc[0]
cool = seg.loc["Cooling off"] if "Cooling off" in seg.index else seg.iloc[-1]
low_rate = item_stats.nsmallest(3, "avg_rating")

print("""\nPORTFOLIO NARRATIVE\n""")
print(f"• {df['customer_id'].nunique()} customers generated ${df['purchase_amount'].sum():,.0f} across {len(df):,} orders.")
print(f"• Average order is ${df['purchase_amount'].mean():.0f}; typical shopper returns ~{df.groupby('customer_id').size().mean():.0f} times.")
print(f"• {top_cat['category']} leads revenue (${top_cat['revenue']:,.0f}, {int(top_cat['orders'])} orders).")
print(f"• Payment split — Credit Card {pay_mix.get('Credit Card', 0)}% / Cash {pay_mix.get('Cash', 0)}% of revenue.")
print(f"• Peak month in the complete series: {best_month['year_month']} (${best_month['revenue']:,.0f}). October 2023 is incomplete.")
print(f"• VIP muses: {int(vip['customers'])} customers, ${vip['revenue']:,.0f} revenue — protect with early drops & styling credits.")
print(f"• Cooling off: {int(cool['customers'])} customers, avg recency {cool['avg_recency']:.0f} days — win-back with category they already love.")
print("• Quality watchlist (lowest ratings):", ", ".join(low_rate["item_purchased"].tolist()))


## 11. Export (optional)



In [ ]:
out_cols = [
    "customer_id", "item_purchased", "category", "purchase_amount", "purchase_date",
    "review_rating", "payment_method", "season", "spend_tier", "rating_band",
    "amount_scale_flag",
]
export = df[out_cols].copy()
export.to_csv("fashion_retail_clean.csv", index=False)
rfm.to_csv("fashion_customer_rfm.csv", index=False)
print("Wrote fashion_retail_clean.csv and fashion_customer_rfm.csv")
export.head()